In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/cell_19/checkpoints/post_cell_20_try_5.pickle

trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['last_ones']
me:  20
trying: ['tqdm']
me:  0
trying: ['sample_submission']
me:  2
trying: ['myword']
me:  12
trying: ['train_first_last']
me:  11
trying: ['len_dict']
me:  12
trying: ['cols_to_display']
me:  14
trying: ['word_dict']
me:  12
trying: ['counter']
me:  11
trying: ['glob']
me:  0
trying: ['mylen']
me:  12
trying: ['ax']
me:  11
trying: ['IREWR_plug_2']
me:  16
trying: ['train_first']
me:  11
trying: ['train_last']
me:  11
trying: ['train']
me:  14
trying: ['stopwords']
me:  1
trying: ['plt']
me:  0
trying: ['add_gap_rows_2']
me:  22
trying: ['style']
me:  0
trying: ['CountVectorizer']
me:  0
trying: ['sample_discourse_id']
me:  2
trying: ['FuncFormatter']
me:  0
trying: ['warnings']
me:  0
trying: ['pd']
me:  0
trying: ['cols_to_merge']
me:  13
trying: ['ax2']
me:  8
trying: ['ax1']
me:  8
trying: ['np']
me:  0
trying: ['orig_output']
me:  21
trying: ['test_txt']
me:  1
trying: ['av_per_e

In [4]:
%%RecordEvent
%%time
### cell 21 ###

# lowercase the discourse text (runs on GPU via the cudf.pandas extension)
train['discourse_text'] = train['discourse_text'].str.lower()

# build the stopword list
stop_english = stopwords.words('english')
other_words_to_take_out = ['school', 'students', 'people', 'would', 'could', 'many']
stop_english.extend(other_words_to_take_out)

# split each discourse_text into words and explode so each word gets its own row
exploded = train.assign(Word=train['discourse_text'].str.split()).explode('Word')

# filter out stopwords (runs on GPU)
filtered = exploded[~exploded['Word'].isin(stop_english)]

# count frequencies of each word by discourse_type
freq = (
    filtered
    .groupby(['discourse_type', 'Word'], sort=False)
    .size()
    .reset_index(name='Frequency')
)

# for each discourse_type, take the top 10 words by Frequency
top10 = (
    freq
    .sort_values(['discourse_type', 'Frequency'], ascending=[True, False])
    .groupby('discourse_type', sort=False)
    .head(10)
)

# build counts_dict from the top10 DataFrame, no conversion to pandas necessary
counts_dict = {}
dt_list = train['discourse_type'].unique().tolist()
for dt in dt_list:
    sub = top10[top10['discourse_type'] == dt]
    df_small = (
        sub
        .drop('discourse_type', axis=1)
        .set_index('Word')
        .sort_values('Frequency', ascending=True)
    )
    counts_dict[dt] = df_small

# extract keys in original order
keys = list(counts_dict.keys())

CPU times: user 123 ms, sys: 64.6 ms, total: 187 ms
Wall time: 195 ms


In [5]:
%Checkpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/cell_19/checkpoints/post_cell_21_try_9.pickle

migration speed (bps): 759177357.7538484
---------------------------
variables to migrate:
cols_to_display 120
IREWR_plug_2 61
train 48
IREWR_plug_1 61
last_ones 48
av_per_essay 48
add_gap_rows 144
np 72
IREWR_tmp2 48
total_gaps 48
IREWR_tmp 48
all_gaps 48
df_small 48
test_txt 120
sub 48
add_gap_rows_2 144
counts_dict 696
glob 144
style 72
i_3 28
CountVectorizer 1072
cols_to_merge 120
tqdm 1072
plt 72
other_words_to_take_out 104
FuncFormatter 1072
warnings 72
exploded 48
sample_discourse_id 32
stopwords 48
dt 57
train_first_last 48
myword 28
i_1 28
len_dict 589920
counter 28
word_dict 589920
stop_english 1912
ax 48
mylen 28
keys 120
train_first 48
myid 61
train_last 48
data 2813
ax1 48
txt_file 208
t 166
ax2 48
orig_output 48
sample_submission 48
freq 48
dt_list 120
train_txt 126104
pd 72
top10 48
filtered 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/notebooks/erikbruin/nlp-on-stu

In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['stopwords', 'train', 'train_txt', 'sample_submission']
Intermediate variables ['test_txt']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['train', 'sample_discourse_id']
Intermediate variables ['sample_submission']
Future variables ['stopwords', 'train_txt']
Modified dataframes
  train
    Input columns: set()
    Changed columns: {'discourse_type_num', 'discourse_text', 'discourse_id', 'discourse_type', 'discourse_end', 'discourse_start', 'id', 'predictionstring'}
    Created columns: set()
    Deleted columns: set()
  sample_submission
    Input columns: set()
    Changed columns: {'predictionstring', 'class', 'id'}
    Created columns: set()
    Deleted columns: set()
======= Cell 2 =======
Input variables []
Active variables ['train', 'cols_to_display']
Intermediate variables []
Future variables ['stopwords', 'train_txt', 'sample_discourse_id']
Modified dataframes
  tra

In [7]:

with open("/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/opt_cell_exec_info_21_try_9.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[21], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
assert False, 'df should be rewritten in the optimized code.'